# NB05 – Descrever e relacionar
**Métodos e Ferramentas de Data Science – FGV MBA · Aula 5 · A teoria, com o código**

## A história
A **Horizonte Distribuidora** (a empresa fictícia do NB01 e do trabalho em grupo) vai vender pela internet, direto para o consumidor. A Marina, gerente de frota, agora cuida também das entregas. Ela achou uma base pública de um e-commerce brasileiro e manda quatro perguntas:

> *"Antes de definir o nosso frete e o nosso prazo, quero aprender com quem já vende online:*
> *1. Quanto custa um frete **típico**?*
> *2. Que cobranças **fogem do padrão**?*
> *3. O frete **acompanha o peso**? E o valor do pedido?*
> *4. O **atraso derruba a nota** que o cliente dá? Se derrubar, vale investir em prazo."*

As duas primeiras perguntas pedem para **descrever uma variável** (o frete). As duas últimas, para **relacionar duas variáveis**, e para não confundir "andam juntos" com "um causa o outro".

| Parte | A pergunta da Marina | A teoria |
|---|---|---|
| 1 | O frete típico | tipo de variável; histograma; média, mediana e moda; desvio, percentis e assimetria |
| 2 | O que foge do padrão | box plot; outliers (regra do IQR e z-score); remover ou limitar |
| 3 | O frete acompanha o peso? | gráfico de dispersão; correlação de Pearson e de Spearman; matriz de correlação |
| 4 | O atraso derruba a nota? | correlação não é causa: as seis armadilhas; comparar dentro de grupos; os critérios de Hill; experimento |

## Os dados
**Olist**, e-commerce brasileiro, pedidos de 2016 a 2018: [Brazilian E-Commerce Public Dataset (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce). Duas tabelas, sem nenhum valor alterado:
- **`itens`** (Partes 1 e 2): um **item vendido** por linha, com preço, frete e peso.
- **`pedidos`** (Partes 3 e 4): um **pedido entregue** por linha, com valor, frete, peso, prazo, atraso e nota.

## Como usar
Execute as células **de cima para baixo**, uma por vez, com **Shift + Enter**.
- **Antes** de cada célula de código, **"O que acontece na célula abaixo"** explica o código.
- **Depois** da saída, **"O que a saída mostra"** traz o resultado e a conclusão. Tente concluir antes de ler.
- Os blocos **Conceito** trazem a teoria, com a referência de onde ela vem.
- Nas **Questões**, escreva o código na célula vazia logo abaixo. Nos blocos **Explore**, mude o código e veja o que acontece; se der erro, desfaça (Ctrl + Z) e execute de novo.

O Python escreve os números com **ponto** decimal (16.26); no texto, usamos vírgula (16,26).

# Parte 1 · O frete típico

## 1. Ferramentas e a primeira base

**O que acontece na célula abaixo**

- Os `import` carregam o **pandas** (tabelas), o **matplotlib** e o **seaborn** (gráficos), como no NB01.
- `pd.set_option(...)` mostra os números com 2 casas decimais. Muda só a forma de mostrar, não o valor guardado.
- `pd.read_csv(...)` lê o arquivo e cria a tabela `itens`. O `.csv.gz` é um CSV compactado; o pandas descompacta sozinho.
- `itens.shape` mostra o tamanho da tabela: **(linhas, colunas)**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.float_format", "{:.2f}".format)

itens = pd.read_csv("https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/olist_itens_aula04.csv.gz")
print(itens.shape)
itens.head()

**O que a saída mostra**

**112.650 linhas e 14 colunas.** Nas três primeiras linhas, a data de entrega (`order_delivered_customer_date`) aparece como `NaN`: o valor **falta**. Na Aula 4 você viu por que um valor falta e quando preencher; aqui o foco é **descrever**.

| Coluna | O que é |
|---|---|
| `order_id`, `order_item_id` | código do pedido e posição do item dentro dele (1, 2, 3...) |
| `order_status` | situação do pedido: `delivered` (entregue), `shipped` (enviado), `canceled` (cancelado) etc. |
| `customer_state` | estado (UF) do cliente |
| `order_purchase_timestamp`, `order_delivered_customer_date`, `order_estimated_delivery_date` | compra, entrega e entrega prometida |
| `product_category_name` | categoria do produto |
| `product_weight_g` e as três medidas em cm | peso e tamanho do produto |
| `price`, `freight_value` | preço e frete **do item**, em R\$ |

## 2. Que tipo de variável é?

> **Conceito · Tipos de variável**
>
> Antes de escolher uma medida ou um gráfico, classifique a variável:
> 
> | Classe | Subtipo | Exemplo nesta base | Medidas e gráficos |
> |---|---|---|---|
> | **Quantitativa** (número que se mede ou conta) | Contínua | `price`, `freight_value`, peso | média, mediana, desvio, percentis; histograma, box plot |
> | | Discreta | `order_item_id`, dias de prazo | contagens, média, mediana |
> | **Qualitativa** (categoria) | Nominal (sem ordem) | `product_category_name`, `customer_state` | frequência; gráfico de barras |
> | | Ordinal (com ordem) | faixa de peso; nota de 1 a 5 | frequência, mediana, percentis |
> 
> **Cuidado:** `order_id` parece um número, mas é um **código**. A "média do código" não significa nada.

Padronizar a unidade e criar faixas ajuda a ler e a comparar. O peso está em gramas; a faixa de peso é uma variável **ordinal** criada a partir de uma contínua.

**O que acontece na célula abaixo**

- `itens["peso_kg"] = itens["product_weight_g"] / 1000` cria a coluna do peso em quilos.
- `pd.cut(...)` corta o peso em **faixas**: `bins` são os limites (0 a 0,5 kg, 0,5 a 2, 2 a 10, 10 a 50) e `labels`, os nomes. São 5 limites para 4 nomes. Em Python, o decimal é com ponto (`0.5`).
- `.value_counts().sort_index()` conta os itens de cada faixa, na ordem dos pesos.

In [ ]:
itens["peso_kg"] = itens["product_weight_g"] / 1000
itens["faixa_peso"] = pd.cut(itens["peso_kg"],
                             bins=[0, 0.5, 2, 10, 50],
                             labels=["até 0,5 kg", "0,5 a 2 kg", "2 a 10 kg", "acima de 10 kg"])
itens["faixa_peso"].value_counts().sort_index()

**O que a saída mostra**

Até 0,5 kg: **47.478** itens; 0,5 a 2 kg: **39.810**; 2 a 10 kg: **20.040**; acima de 10 kg: **5.296**. Quase metade dos itens pesa até meio quilo. Os itens sem peso, ou com peso zero (que não existe), ficam sem faixa: o primeiro limite é "acima de 0".

## 3. A forma: o histograma

> **Conceito · Histograma**
>
> O **histograma** divide os valores em intervalos e mostra, em cada barra, **quantos** caem ali. É o primeiro gráfico de uma variável quantitativa: mostra onde está o miolo, se há caudas e se há valores isolados. Na **escala logarítmica**, cada marca do eixo vale 10 vezes a anterior: os valores pequenos e os grandes cabem no mesmo gráfico. Ela não aceita o zero.

![Como se constrói um histograma: cada pedido cai numa faixa de valores, e a barra conta quantos caíram ali (números hipotéticos).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_histograma.png)

*Como se constrói um histograma: cada pedido cai numa faixa de valores, e a barra conta quantos caíram ali (números hipotéticos).*

**O que acontece na célula abaixo**

- `frete = itens["freight_value"]` dá um nome curto à coluna do frete.
- `plt.subplots(1, 2, ...)` abre dois gráficos lado a lado: `ax[0]` à esquerda, `ax[1]` à direita.
- `sns.histplot(frete, bins=80, ax=ax[0])` desenha o histograma com 80 barras. À direita, `frete[frete > 0]` tira os fretes zero e `log_scale=True` usa a escala logarítmica.

In [ ]:
frete = itens["freight_value"]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(frete, bins=80, ax=ax[0])
ax[0].set_title("Frete por item (R$)")
sns.histplot(frete[frete > 0], bins=80, log_scale=True, ax=ax[1])
ax[1].set_title("Frete por item, escala logarítmica")
plt.show()

**O que a saída mostra**

À esquerda, quase tudo encostado no começo do eixo (fretes de R\$ 10 a R\$ 25) e uma **cauda longa à direita**, que vai até perto de R\$ 400. À direita, a mesma informação, com a forma mais fácil de ver. Uma cauda assim avisa: **a média vai ser puxada para cima.**

> **Explore.** Troque `bins=80` por `bins=20` e depois por `bins=200`. A forma muda? Com poucas barras, o histograma esconde detalhes; com muitas, vira ruído.

## 4. Medidas de posição: média, mediana e moda

> **Conceito · Média, mediana e moda**
>
> - **Média** = soma dos valores ÷ quantidade: $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$. Usa **o valor** de cada linha: um único valor extremo pode mudá-la muito.
> - **Mediana** = o valor do meio, com os dados em ordem: metade fica abaixo, metade acima. Usa **a posição**: resiste a valores extremos.
> - **Moda** = o valor mais frequente. Serve também para variáveis qualitativas.
> 
> Numa distribuição simétrica, média e mediana ficam próximas. Com cauda à direita, a **média fica acima da mediana** (é a renda típica da sala, na abertura da Aula 4).

![Média e mediana com 7 fretes: R\$ 12, 14, 15, 15, 16, 18 e 22. Soma 112 ÷ 7 = média 16; em ordem, o 4º valor (o do meio) é a mediana, 15; a moda é 15. Trocando R\$ 22 por R\$ 120, a média vai a 30 e a mediana continua 15 (números hipotéticos).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_media_mediana.png)

*Média e mediana com 7 fretes: R\$ 12, 14, 15, 15, 16, 18 e 22. Soma 112 ÷ 7 = média 16; em ordem, o 4º valor (o do meio) é a mediana, 15; a moda é 15. Trocando R\$ 22 por R\$ 120, a média vai a 30 e a mediana continua 15 (números hipotéticos).*

**O que acontece na célula abaixo**

- `.mean()`, `.median()` e `.mode()` calculam média, mediana e moda. A moda vem como lista (pode haver empate); o `[0]` pega o primeiro valor.
- `print(f"...")` monta o texto: o que está entre `{ }` é trocado pelo valor, e `:.2f` mostra 2 casas.

In [ ]:
print(f"Média:   R$ {frete.mean():.2f}")
print(f"Mediana: R$ {frete.median():.2f}")
print(f"Moda:    R$ {frete.mode()[0]:.2f}")

**O que a saída mostra**

Média de **R\$ 19,99**, mediana de **R\$ 16,26** e moda de **R\$ 15,10**. A média está acima da mediana: os poucos fretes altos puxam a média. Para a **primeira pergunta da Marina** ("quanto custa um frete típico?"), a **mediana** representa melhor o cliente comum.

## 5. Dispersão, percentis e assimetria

> **Conceito · Medidas de dispersão**
>
> Saber o valor típico não basta: é preciso saber **o quanto os valores variam**.
> - **Desvio padrão**: o afastamento típico em relação à média, na mesma unidade dos dados: $s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2}$. Como a média, sofre com extremos.
> - **Coeficiente de variação** = desvio padrão ÷ média: compara a variação de coisas em escalas diferentes.
> - **Percentil** $p$: o valor abaixo do qual ficam $p$% dos dados. O P50 é a mediana; o P25 e o P75 são o **1º e o 3º quartis** (Q1 e Q3).
> - **Amplitude interquartil** (IQR) = Q3 − Q1: o tamanho do "miolo" com os 50% do meio. Resiste a extremos.
> - **Assimetria**: 0 numa distribuição simétrica; positiva quando a cauda vai para a direita; negativa quando vai para a esquerda.
> 
> Percentis viram regras que a operação entende: "90% dos fretes custam até X".

![Duas lojas com o mesmo prazo médio (16 dias) e espalhamentos diferentes. Na loja A, as distâncias até a média são −2, −1, 0, 1 e 2; ao quadrado, 4, 1, 0, 1 e 4; a soma, 10, dividida por n − 1 = 4, dá 2,5, e a raiz, 1,6 dia. Na loja B, as mesmas contas dão 7,9 dias (números hipotéticos).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_desvio.png)

*Duas lojas com o mesmo prazo médio (16 dias) e espalhamentos diferentes. Na loja A, as distâncias até a média são −2, −1, 0, 1 e 2; ao quadrado, 4, 1, 0, 1 e 4; a soma, 10, dividida por n − 1 = 4, dá 2,5, e a raiz, 1,6 dia. Na loja B, as mesmas contas dão 7,9 dias (números hipotéticos).*

![As três formas mais comuns e onde ficam a média e a mediana em cada uma (dados simulados).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_formas.png)

*As três formas mais comuns e onde ficam a média e a mediana em cada uma (dados simulados).*

**O que acontece na célula abaixo**

- `frete.quantile([0.25, 0.75])` calcula Q1 e Q3; `q1, q3 = ...` guarda um em cada nome. `iqr = q3 - q1`.
- `.std()` é o desvio padrão; `:.0%` mostra um número como porcentagem.
- `.skew()` mede a assimetria.
- A última linha calcula seis percentis de uma vez.

In [ ]:
q1, q3 = frete.quantile([0.25, 0.75])
iqr = q3 - q1
print(f"Desvio padrão:            R$ {frete.std():.2f}")
print(f"Amplitude interquartil:   R$ {iqr:.2f}")
print(f"Coeficiente de variação:  {frete.std() / frete.mean():.0%}")
print(f"Assimetria:               {frete.skew():.2f}")
print()
frete.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

**O que a saída mostra**

Desvio padrão de **R\$ 15,81**, IQR de **R\$ 8,07**, coeficiente de variação de **79%** e assimetria de **5,64** (cauda longa à direita). Percentis: P10 **8,73** · P25 **13,08** · P50 **16,26** · P75 **21,15** · P90 **34,04** · P99 **84,52**.

Leitura para a Marina: metade dos fretes fica entre R\$ 13 e R\$ 21; 90% custam até R\$ 34; só 1% passa de R\$ 85. O desvio (15,81) é quase o dobro do IQR (8,07): os extremos inflam o desvio, não o IQR.

**Questão 1.** O **preço** (`price`) tem a mesma forma do frete? Calcule a média, a mediana e a assimetria.
Dica: `itens["price"].mean()`, `.median()` e `.skew()`.

In [ ]:
# Questão 1: escreva aqui


# Parte 2 · O que foge do padrão

## 6. Box plot: a distribuição numa figura

> **Conceito · Box plot (Tukey, 1977)**
>
> O **box plot** resume a distribuição: a **caixa** vai do Q1 ao Q3; a linha dentro dela é a **mediana**; as **hastes** vão até o último valor dentro de **1,5 × IQR** além da caixa; o que passa disso aparece como **ponto**. É bom para **comparar grupos** lado a lado. Proposto por Tukey, J. W. *Exploratory Data Analysis*. Addison-Wesley, 1977.

![Como ler um box plot: embaixo, cada ponto cinza é um pedido; em cima, o resumo deles (dados simulados).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_boxplot.png)

*Como ler um box plot: embaixo, cada ponto cinza é um pedido; em cima, o resumo deles (dados simulados).*

**O que acontece na célula abaixo**

- `sns.boxplot(data=itens, x="faixa_peso", y="freight_value", ...)` desenha um box plot do frete para cada faixa de peso. O seaborn agrupa sozinho, como um `groupby`.
- `showfliers=False` esconde os pontos extremos; com eles, as caixas ficariam espremidas no fundo.

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=itens, x="faixa_peso", y="freight_value", showfliers=False)
plt.title("Frete por faixa de peso (sem os pontos extremos)")
plt.ylabel("Frete (R$)"); plt.xlabel("")
plt.show()

**O que a saída mostra**

As caixas **sobem** da esquerda para a direita: quanto mais pesado, mais caro o frete. A caixa acima de 10 kg é bem mais alta e mais **larga**: além de mais caro, o frete dos itens pesados **varia** muito mais. Um frete de R\$ 60 é comum para um item de 20 kg e raro para um de 300 g: **o que é "fora do padrão" depende do grupo.**

> **Explore.** Troque `showfliers=False` por `showfliers=True`. Quantos pontos aparecem acima das hastes? Eles são erro?

## 7. Outliers: erro, fraude ou exceção legítima?

> **Conceito · Duas regras para apontar outliers**
>
> - **Regra do IQR** (a do box plot, Tukey, 1977): fora de $[Q_1 - 1{,}5 \times IQR;\; Q_3 + 1{,}5 \times IQR]$.
> - **Z-score**: quantos desvios padrão o valor está longe da média, $z = \frac{x - \bar{x}}{s}$; costuma-se marcar $|z| > 3$. Funciona mal com cauda longa, porque a média e o desvio já foram puxados pelos extremos.
> 
> A regra diz **onde olhar**; o contexto diz **o que é**. Um outlier pode ser **erro** (digitação, unidade errada), **fraude** ou uma **exceção legítima** (um produto de 30 kg enviado para longe). Outlier é uma pergunta, não uma resposta.

![A regra do IQR passo a passo, com 11 fretes: Q1 = 12,5, mediana = 15, Q3 = 19,5; IQR = 19,5 − 12,5 = 7; limite = 19,5 + 1,5 × 7 = 30. O frete de R\$ 40 passa do limite: é um candidato a outlier (números hipotéticos).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_quartis.png)

*A regra do IQR passo a passo, com 11 fretes: Q1 = 12,5, mediana = 15, Q3 = 19,5; IQR = 19,5 − 12,5 = 7; limite = 19,5 + 1,5 × 7 = 30. O frete de R\$ 40 passa do limite: é um candidato a outlier (números hipotéticos).*

**O que acontece na célula abaixo**

- `lim_sup = q3 + 1.5 * iqr` é o limite superior da regra do IQR (o `q3` e o `iqr` vêm da seção 5).
- `z = (frete - frete.mean()) / frete.std()` calcula o z-score de cada frete; `z.abs()` tira o sinal.
- `(condição).sum()` conta quantos passam; `(condição).mean()` dá a proporção.

In [ ]:
lim_sup = q3 + 1.5 * iqr
z = (frete - frete.mean()) / frete.std()
print(f"Limite superior (regra do IQR): R$ {lim_sup:.2f}")
print(f"Acima do limite do IQR: {(frete > lim_sup).sum()} itens ({(frete > lim_sup).mean():.1%})")
print(f"Z-score acima de 3:     {(z.abs() > 3).sum()} itens ({(z.abs() > 3).mean():.1%})")

**O que a saída mostra**

Pela regra do IQR, o limite é **R\$ 33,25**, e **11.613 itens (10,3%)** passam dele. Pelo z-score, **2.041 (1,8%)**. As duas regras discordam muito: com uma cauda longa, 10% "fora do padrão" é muita coisa para revisar um a um.

**O que acontece na célula abaixo**

- `itens.sort_values("freight_value", ascending=False)` ordena do maior frete para o menor; `[[...]]` escolhe as colunas que ajudam a julgar cada caso; `.head(8)` mostra os 8 primeiros.

In [ ]:
itens.sort_values("freight_value", ascending=False)[["product_category_name", "peso_kg", "customer_state", "price", "freight_value"]].head(8)

**O que a saída mostra**

O maior frete é de **R\$ 409,68**, num produto de bebê de 14,7 kg enviado ao Piauí. Vários dos seguintes são produtos de 25 a 30 kg, com preço acima de R\$ 900. Parecem **exceções legítimas** (pesados e caros), não erro. É a **segunda pergunta da Marina**: a regra aponta; a investigação decide.

> **Conceito · Remover ou limitar (Galli, 2020)**
>
> Depois de investigar, duas saídas comuns, cada uma com um custo (Galli, S. *Python Feature Engineering Cookbook*. Packt, 2020):
> - **Remover** (*trimming*): tira as linhas da análise. Perde informação; a base encolhe.
> - **Limitar** (*capping*): troca o valor extremo pelo limite. Mantém as linhas, mas altera os valores.
> 
> Toda decisão vai para o registro da análise, porque muda os números.

**O que acontece na célula abaixo**

- `frete[frete <= lim_sup]` é um filtro: fica só com os fretes até o limite.
- `frete.clip(upper=lim_sup)` mantém todos, mas troca os que passam do limite pelo próprio limite.
- `pd.DataFrame({...})` põe o `.describe()` das três versões lado a lado.

In [ ]:
removendo = frete[frete <= lim_sup]
limitando = frete.clip(upper=lim_sup)
pd.DataFrame({"original": frete.describe(),
              "removendo": removendo.describe(),
              "limitando": limitando.describe()})

**O que a saída mostra**

Removendo, sobram **101.037** itens e a média cai para **16,04**; limitando, ficam os **112.650** e a média vai para **17,82**. A **mediana** quase não se mexe (15,58 e 16,26). Para descrever o frete típico, a mediana já resiste aos extremos **sem apagar nada**.

**Questão 2.** Calcule a **mediana do frete em cada faixa de peso**. Quantas vezes o frete mediano acima de 10 kg é maior que o de até 0,5 kg?
Dica: `itens.groupby("faixa_peso", observed=True)["freight_value"].median()`.

In [ ]:
# Questão 2: escreva aqui


# Parte 3 · O frete acompanha o peso?

## 8. A segunda base: um pedido por linha

**O que acontece na célula abaixo**

- `pd.read_csv(...)` lê a tabela `pedidos`: só pedidos **entregues**, um por linha, com o valor, o frete e o peso somados por pedido, o prazo, o atraso e a nota. Cada linha da tabela `itens` é um item; para perguntar sobre **pedidos**, a tabela precisa ter uma linha por pedido (Aula 4).

In [ ]:
pedidos = pd.read_csv("https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/olist_pedidos_aula05.csv.gz")
print(pedidos.shape)
pedidos.head()

**O que a saída mostra**

**96.470 pedidos** e 12 colunas.

| Coluna | O que é |
|---|---|
| `uf`, `regiao` | estado e região do cliente |
| `itens`, `valor`, `frete` | número de itens, soma dos preços e soma dos fretes do pedido (R\$) |
| `peso_kg` | peso total do pedido |
| `prazo_dias` | dias entre a compra e a entrega |
| `atraso_dias`, `atrasou` | dias entre a data prometida e a entrega (negativo = adiantado); `True` se chegou depois do prometido |
| `nota` | nota da avaliação, de 1 a 5 (vazia se o cliente não avaliou) |

## 9. Primeiro o gráfico, depois o número

> **Conceito · O quarteto de Anscombe (1973)**
>
> Anscombe montou quatro conjuntos de 11 pontos com a **mesma média** e a **mesma correlação (0,82)**, mas formas completamente diferentes: uma reta com ruído, uma curva, uma reta com um ponto fora, e uma nuvem vertical com um único ponto distante que cria a correlação inteira (slide da aula). Moral: **a correlação resume a nuvem; não a substitui.** Anscombe, F. J. Graphs in statistical analysis. *The American Statistician*, 27(1), 1973.

![Como ler a correlação: o sinal diz a direção, e o tamanho diz quão perto de uma reta os pontos ficam. O último painel tem uma relação forte, mas em curva, e r perto de zero (dados simulados).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_correlacoes.png)

*Como ler a correlação: o sinal diz a direção, e o tamanho diz quão perto de uma reta os pontos ficam. O último painel tem uma relação forte, mas em curva, e r perto de zero (dados simulados).*

**O que acontece na célula abaixo**

- `pedidos.dropna(subset=["peso_kg"])` tira os pedidos sem peso; `.sample(3000, random_state=42)` sorteia 3.000 deles (com 96 mil, os pontos se amontoam). O `random_state` fixa o sorteio: todos veem os mesmos pontos.
- `sns.scatterplot(...)` desenha um ponto por pedido; `alpha=0.3` deixa os pontos transparentes.
- `plt.xscale("log")` e `plt.yscale("log")` usam a escala logarítmica nos dois eixos.

In [ ]:
amostra = pedidos.dropna(subset=["peso_kg"]).sample(3000, random_state=42)

plt.figure(figsize=(7, 5))
sns.scatterplot(data=amostra, x="peso_kg", y="frete", alpha=0.3, s=14)
plt.xscale("log"); plt.yscale("log")
plt.title("Frete × peso (amostra de 3.000 pedidos)")
plt.xlabel("Peso do pedido (kg)"); plt.ylabel("Frete do pedido (R$)")
plt.show()

**O que a saída mostra**

A nuvem **sobe** da esquerda para a direita: pedidos mais pesados pagam mais frete. Mas ela é **larga**: pedidos do mesmo peso pagam fretes bem diferentes. A distância também conta, e ela não está nesta base.

## 10. Pearson e Spearman

> **Conceito · Correlação de Pearson (1896)**
>
> Mede a força da relação **em linha reta** entre duas variáveis quantitativas:
> 
> $$r = \frac{\sum_{i}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i}(x_i - \bar{x})^2}\;\sqrt{\sum_{i}(y_i - \bar{y})^2}}$$
> 
> Vai de **−1** (um sobe, o outro desce, numa reta perfeita) a **+1** (sobem juntos, numa reta perfeita); **0** = não há relação **em linha reta** (pode haver outra forma, como uma curva). Como usa os **valores** e as **médias**, é sensível a valores extremos. Pearson, K. Mathematical contributions to the theory of evolution. III. Regression, heredity, and panmixia. *Philosophical Transactions of the Royal Society A*, 187, 1896.

> **Conceito · Correlação de Spearman (1904)**
>
> É a correlação de Pearson calculada sobre as **posições** (postos) dos valores: o menor vira 1, o seguinte 2, e assim por diante. Sem empates, fica
> 
> $$\rho = 1 - \frac{6\sum_{i} d_i^2}{n(n^2 - 1)}$$
> 
> em que $d_i$ é a diferença entre as posições de $x_i$ e de $y_i$. Mede se as duas variáveis **sobem (ou descem) juntas**, mesmo que não numa reta (relação **monotônica**). Como usa a ordem, um valor extremo vale só "o último da fila". Spearman, C. The proof and measurement of association between two things. *The American Journal of Psychology*, 15(1), 1904.
> 
> | Use **Pearson** quando | Use **Spearman** quando |
> |---|---|
> | as duas variáveis são quantitativas | uma delas é **ordinal** (nota de 1 a 5, faixa, ranking) |
> | o gráfico mostra uma relação em **linha reta** | a relação sobe sempre, mas **curva** |
> | não há valores extremos que dominem a nuvem | há **caudas longas** ou valores extremos |
> 
> Se os dois discordam muito, **olhe o gráfico**: provavelmente há extremos ou uma relação que não é reta. Nenhum dos dois diz **por que** as variáveis andam juntas.

![Pearson × Spearman. À esquerda, valores que sobem sempre, mas em curva: o Spearman vale 1 e o Pearson, 0,84. À direita, uma nuvem sem relação e um único ponto extremo: o ponto puxa o Pearson para 0,58; o Spearman fica perto de zero (dados hipotéticos e simulados).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_pearson_spearman.png)

*Pearson × Spearman. À esquerda, valores que sobem sempre, mas em curva: o Spearman vale 1 e o Pearson, 0,84. À direita, uma nuvem sem relação e um único ponto extremo: o ponto puxa o Pearson para 0,58; o Spearman fica perto de zero (dados hipotéticos e simulados).*

**O que acontece na célula abaixo**

- `pedidos["frete"].corr(pedidos["peso_kg"])` calcula a correlação de **Pearson** (é o padrão do pandas).
- `method="spearman"` troca para **Spearman**.
- A segunda parte repete as contas **sem o 1% de fretes mais altos**: `quantile(0.99)` dá o limite, e o filtro fica com os fretes até ele.

In [ ]:
print(f"Todos os pedidos: Pearson {pedidos['frete'].corr(pedidos['peso_kg']):.2f} | "
      f"Spearman {pedidos['frete'].corr(pedidos['peso_kg'], method='spearman'):.2f}")

limite = pedidos["frete"].quantile(0.99)
sem_extremos = pedidos[pedidos["frete"] <= limite]
print(f"Sem o 1% maior:   Pearson {sem_extremos['frete'].corr(sem_extremos['peso_kg']):.2f} | "
      f"Spearman {sem_extremos['frete'].corr(sem_extremos['peso_kg'], method='spearman'):.2f}")

**O que a saída mostra**

Com todos os pedidos, **Pearson 0,64 e Spearman 0,49**. Sem o 1% de fretes mais altos, **Pearson cai para 0,54** e **Spearman quase não muda (0,48)**. O 1% de extremos inflava o Pearson; o Spearman, que usa a ordem, resiste. A **terceira pergunta da Marina**: sim, o frete acompanha o peso, de forma **moderada**; o peso não explica tudo.

**Questão 3.** A `nota` vai de 1 a 5. Que correlação faz mais sentido entre `prazo_dias` e `nota`? Calcule as duas.
Dica: `pedidos["prazo_dias"].corr(pedidos["nota"], method="spearman")`.

In [ ]:
# Questão 3: escreva aqui


## 11. Várias correlações de uma vez

**O que acontece na célula abaixo**

- `pedidos[colunas].corr(method="spearman")` calcula a correlação de Spearman de **cada par** de colunas: a **matriz de correlação**.
- `sns.heatmap(...)` pinta a matriz: `annot=True` escreve os números; `vmin=-1, vmax=1` fixa a escala; `cmap="RdBu_r"` usa azul para positivo e vermelho para negativo.

In [ ]:
colunas = ["valor", "frete", "peso_kg", "itens", "prazo_dias", "nota"]
matriz = pedidos[colunas].corr(method="spearman")

plt.figure(figsize=(7, 5.5))
sns.heatmap(matriz, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="RdBu_r")
plt.title("Correlação de Spearman entre as colunas")
plt.show()

**O que a saída mostra**

Os pares mais fortes são **valor × peso (0,54)**, **frete × peso (0,49)** e **valor × frete (0,47)**. **Prazo × nota** é negativa (**−0,23**). A diagonal é sempre 1 (cada coluna com ela mesma). Um número forte aqui é uma **pista**, não uma conclusão: valor e frete andam juntos, mas um não causa o outro, como a Parte 4 mostra.

# Parte 4 · O atraso derruba a nota?

## 12. Correlação não é causa

> **Conceito · As seis armadilhas, com o nome da referência**
>
> | Armadilha | O que acontece | Referência |
> |---|---|---|
> | **Fator de confusão** (*confounding*) | um terceiro fator puxa as duas variáveis | Freedman, Pisani e Purves, *Statistics*, 4. ed., 2007, cap. 2 (estudos observacionais) |
> | **Causalidade reversa** | a seta aponta ao contrário: o "efeito" veio antes. Hill chama o critério de *temporality*: a causa precisa vir antes do efeito | Hill (1965) |
> | **Viés de sobrevivência** (*survivorship bias*) | só vemos quem ficou na base; quem saiu levava a outra metade da história | Brown, Goetzmann, Ibbotson e Ross (1992) |
> | **Regressão à média** (*regression towards mediocrity*; hoje, *regression to the mean*) | quem foi escolhido por estar no extremo tende a voltar para perto da média na medida seguinte, sem nenhuma intervenção | Galton (1886); Barnett, van der Pols e Dobson (2005) |
> | **Correlação sem sentido** (*nonsense-correlation*) | duas séries que crescem (ou caem) com o tempo parecem relacionadas sem ter nada em comum | Yule (1926) |
> | **Viés de seleção amostral** (*sample selection bias*) | quem entra no grupo não é sorteado: quem escolhe (ou é escolhido) já é diferente; a autosseleção é o caso mais comum | Heckman (1979) |
> 
> Os cases da segunda metade da aula têm uma armadilha cada.

![As seis armadilhas em diagramas. Em laranja, o que a diretoria não vê (diagramas e dados simulados).](https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/notebooks/img/t_armadilhas.png)

*As seis armadilhas em diagramas. Em laranja, o que a diretoria não vê (diagramas e dados simulados).*

Comece pelo fator de confusão. Valor e frete andam juntos (seção 11). Cobrar mais frete faz o cliente gastar mais? Nenhum dos dois: produto **pesado** costuma ser **mais caro** e pagar **mais frete**. Se o peso puxa os dois, a correlação deve cair quando comparamos pedidos de **peso parecido**.

> **Conceito · Comparar dentro de grupos**
>
> A primeira defesa contra o fator de confusão é **comparar dentro de grupos parecidos** (estratificar): se a relação some dentro de cada grupo, ela vinha do fator que define os grupos (Freedman, Pisani e Purves, 2007, cap. 2).

**O que acontece na célula abaixo**

- `pd.cut(...)` cria as faixas de peso dos pedidos, como na seção 2 (o último limite vai a 1.000 kg).
- O `for faixa, grupo in pedidos.groupby(...)` percorre as faixas: em cada volta, `grupo` é a tabela só com os pedidos daquela faixa, e o `print` mostra a correlação de Pearson dentro dela.

In [ ]:
pedidos["faixa_peso"] = pd.cut(pedidos["peso_kg"], bins=[0, 0.5, 2, 10, 1000],
                               labels=["até 0,5 kg", "0,5 a 2 kg", "2 a 10 kg", "acima de 10 kg"])

print(f"Valor × frete, todos os pedidos: {pedidos['valor'].corr(pedidos['frete']):.2f}")
for faixa, grupo in pedidos.groupby("faixa_peso", observed=True):
    print(f"  dentro da faixa {faixa:>14}: {grupo['valor'].corr(grupo['frete']):.2f}   ({len(grupo)} pedidos)")

**O que a saída mostra**

No total, **0,41**; dentro das faixas, **0,23, 0,23, 0,33 e 0,37**. Boa parte da relação entre valor e frete **era o peso**. A correlação não some: dentro da faixa ainda sobram tamanho, distância e categoria. A pergunta diante de toda correlação: **o que mais anda junto com as duas?**

## 13. Atraso e nota: os critérios de Hill

**O que acontece na célula abaixo**

- `pedidos.groupby("atrasou")["nota"].agg(["mean", "count"])` calcula a nota média e quantos pedidos com nota há em cada grupo: no prazo (`False`) e atrasados (`True`).

In [ ]:
pedidos.groupby("atrasou")["nota"].agg(["mean", "count"])

**O que a saída mostra**

No prazo, nota **4,29**; atrasados, **2,27**: uma queda de **2 pontos**, com 89.443 e 6.381 pedidos avaliados. A associação é **forte**. Mas é causa? E se for a região: Norte e Nordeste têm prazos maiores e outro perfil de compra?

> **Conceito · Os critérios de Hill (1965)**
>
> Bradford Hill propôs nove pontos de vista para avaliar se uma associação é causa: **força** (*strength*), **consistência** (*consistency*, repete-se em outros grupos, lugares e tempos), **especificidade** (*specificity*), **temporalidade** (*temporality*, a causa vem antes), **gradiente** (*biological gradient*, mais exposição, mais efeito), **plausibilidade** (*plausibility*, há um mecanismo), **coerência** (*coherence*), **experimento** (*experiment*) e **analogia** (*analogy*). Para Hill, nenhum deles prova a causa sozinho, e nenhum é obrigatório: juntos, aumentam ou diminuem a confiança. Hill, A. B. The environment and disease: association or causation? *Proceedings of the Royal Society of Medicine*, 58(5), 295–300, 1965.
> 
> As duas próximas células testam dois deles: **consistência** (a queda aparece em todas as regiões?) e **gradiente** (quanto mais atraso, menor a nota?).

**O que acontece na célula abaixo**

- `pivot_table(...)` monta uma tabela com as regiões nas linhas, no prazo e atrasado nas colunas, e a nota média em cada cruzamento.
- `tabela.columns = [...]` dá nomes às duas colunas; a coluna `diferença` subtrai uma da outra.

In [ ]:
tabela = pedidos.pivot_table(index="regiao", columns="atrasou", values="nota", aggfunc="mean")
tabela.columns = ["no prazo", "atrasado"]
tabela["diferença"] = tabela["atrasado"] - tabela["no prazo"]
tabela

**O que a saída mostra**

A queda de cerca de **2 pontos aparece em todas as regiões** (de −1,93 no Sul a −2,08 no Centro-Oeste). A região não explica a queda: a associação é **consistente**.

**O que acontece na célula abaixo**

- `pd.cut(pedidos["atraso_dias"], ...)` cria faixas de atraso: até 0 dia (no prazo ou adiantado), 1 a 3, 4 a 7, 8 a 14 e 15 ou mais.
- `groupby(...)["nota"].agg(["mean", "count"])` dá a nota média e o número de pedidos avaliados em cada faixa.

In [ ]:
pedidos["faixa_atraso"] = pd.cut(pedidos["atraso_dias"], bins=[-1000, 0, 3, 7, 14, 1000],
                                 labels=["no prazo", "1 a 3 dias", "4 a 7 dias", "8 a 14 dias", "15 dias ou mais"])
pedidos.groupby("faixa_atraso", observed=True)["nota"].agg(["mean", "count"])

**O que a saída mostra**

A nota cai de **4,29** (no prazo) para **3,29** com 1 a 3 dias de atraso, **2,11** com 4 a 7 dias e cerca de **1,7** a partir de 8 dias: quanto mais atraso, menor a nota, até um piso. É o **gradiente** de Hill (mais exposição, mais efeito). Somando: a associação é **forte**, **consistente** entre regiões, tem **gradiente**, respeita a **temporalidade** (o cliente avalia depois de receber) e é **plausível**. Ainda **não é prova**: vendedor, produto e embalagem podem andar junto com o atraso. Mas é uma evidência bem melhor do que um número sozinho. A **quarta pergunta da Marina**: investir em prazo deve proteger a nota.

> **Conceito · O experimento**
>
> Para afirmar causa, o padrão é o **experimento**: **sortear** quem recebe a mudança e comparar com quem não recebeu. O sorteio deixa os dois grupos parecidos em tudo, inclusive no que não foi medido (Freedman, Pisani e Purves, 2007, cap. 1, experimentos controlados). Quando não dá para sortear (atrasar de propósito não é uma opção), chegue perto: compare grupos parecidos, compare antes e depois com um grupo de controle, e veja se o resultado se repete em vários grupos.

**Questão 4.** Que **porcentagem dos pedidos atrasa** em cada região? A região que mais atrasa é a que perde mais nota com o atraso?
Dica: a média de uma coluna `True`/`False` é a proporção de `True`: `pedidos.groupby("regiao")["atrasou"].mean()`.

In [ ]:
# Questão 4: escreva aqui


> **Explore.** Na célula da faixa de atraso, troque `"nota"` por `"valor"`. Pedidos atrasados são mais caros ou mais baratos? Isso poderia ser um fator de confusão?

---
## Resumo
- **Olhe a forma** (histograma) antes de resumir num número. Com cauda longa, a **mediana** e o **IQR** representam melhor que a média e o desvio.
- **Outlier é uma pergunta.** A regra do IQR diz onde olhar; o contexto diz se é erro, fraude ou exceção legítima. Remover ou limitar muda os números: registre.
- **Primeiro o gráfico, depois a correlação.** Pearson mede linha reta e sofre com extremos; Spearman usa a ordem e serve para variáveis ordinais.
- **Correlação é pista, não prova.** Pergunte o que mais anda junto, compare dentro de grupos, use os critérios de Hill e, se puder, sorteie.

## Referências
- ANSCOMBE, F. J. Graphs in statistical analysis. *The American Statistician*, 27(1), 1973.
- BARNETT, A. G.; VAN DER POLS, J. C.; DOBSON, A. J. Regression to the mean: what it is and how to deal with it. *International Journal of Epidemiology*, 34(1), 215–220, 2005.
- BROWN, S. J.; GOETZMANN, W.; IBBOTSON, R. G.; ROSS, S. A. Survivorship bias in performance studies. *The Review of Financial Studies*, 5(4), 553–580, 1992.
- FREEDMAN, D.; PISANI, R.; PURVES, R. *Statistics*. 4. ed. W. W. Norton, 2007.
- GALLI, S. *Python Feature Engineering Cookbook*. Packt, 2020.
- GALTON, F. Regression towards mediocrity in hereditary stature. *Journal of the Anthropological Institute*, 15, 1886.
- HECKMAN, J. J. Sample selection bias as a specification error. *Econometrica*, 47(1), 153–161, 1979.
- HILL, A. B. The environment and disease: association or causation? *Proceedings of the Royal Society of Medicine*, 58(5), 295–300, 1965.
- PEARSON, K. Mathematical contributions to the theory of evolution. III. Regression, heredity, and panmixia. *Philosophical Transactions of the Royal Society A*, 187, 1896.
- SPEARMAN, C. The proof and measurement of association between two things. *The American Journal of Psychology*, 15(1), 1904.
- TUKEY, J. W. *Exploratory Data Analysis*. Addison-Wesley, 1977.
- YULE, G. U. Why do we sometimes get nonsense-correlations between time-series? *Journal of the Royal Statistical Society*, 89(1), 1–63, 1926.

## Erros comuns e como resolver

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `NameError: name 'itens'` (ou `'pedidos'`, `'frete'`, `'q3'`) `is not defined` | A célula que cria esse nome não foi executada | Execute as células anteriores (*Ambiente de execução → Executar anteriores*) |
| `KeyError: 'faixa_peso'` | A célula que cria a faixa não foi executada | Execute a seção 2 (itens) ou a 12 (pedidos) |
| `KeyError: 'Price'` | Nome da coluna com letra diferente | Os nomes estão em minúsculas: `"price"` |
| `ValueError: could not convert string to float` no `.corr()` | Há colunas de texto na tabela | Selecione só colunas numéricas, como na seção 11 |
| Aviso `FutureWarning: ... observed=False` | Mudança de comportamento do pandas | Pode ignorar, ou use `observed=True` no `groupby` |
| `HTTPError` ou `URLError` | Sem internet ou rede bloqueada | Use o Plano B abaixo |

## Plano B: se o download falhar
1. Baixe os arquivos `olist_itens_aula04.csv.gz` e `olist_pedidos_aula05.csv.gz` da pasta `dados/` do repositório da disciplina (ou pelo link do ECLASS).
2. No Colab, clique no ícone de **pasta** na barra lateral esquerda e arraste os dois arquivos para lá.
3. Nas células de leitura (seções 1 e 8), apague o endereço que vem antes do nome do arquivo, deixando só `"olist_itens_aula04.csv.gz"` e `"olist_pedidos_aula05.csv.gz"`, e execute de novo.

### Autor e licença

**Autor:** Prof. Carlos Carlim · [LinkedIn](https://www.linkedin.com/in/carloscarlim-77144120) · [GitHub](https://github.com/carloscarlim)

**Licença MIT.** Copyright (c) 2026 Carlos Carlim. Os direitos autorais deste material pertencem exclusivamente ao autor.

É permitido usar, copiar, modificar, mesclar, publicar, distribuir, sublicenciar e vender cópias deste material, desde que este aviso de copyright e o texto da licença sejam mantidos em todas as cópias ou partes substanciais. O material é fornecido "no estado em que se encontra", sem garantia de qualquer tipo. Texto completo da licença: [LICENSE](https://github.com/carloscarlim/fgv-mfds/blob/main/LICENSE).

Os dados usados nos notebooks têm licenças próprias, indicadas em [`dados/README.md`](https://github.com/carloscarlim/fgv-mfds/blob/main/dados/README.md).